# ========================================
# VENDOR ANALYSIS PROJECT - PYSPARK BIG DATA PROCESSING
# Notebook 4: Advanced Analytics with PySpark
# ========================================

In [10]:
# CELL 1: Setup and Install PySpark
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyspark

import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')

print("✅ Setup complete\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete



In [12]:
# CELL 2: Initialize PySpark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window
import pandas as pd
import os
import glob
from pyspark.errors import PySparkRuntimeError

# Create Spark session with optimized settings
try:
    spark = SparkSession.builder \
        .appName("VendorBigDataAnalysis") \
        .config("spark.driver.memory", "6g") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    print("✅ Spark session created")
    print(f"   Spark version: {spark.version}")
    print(f"   Master: {spark.sparkContext.master}\n")

except PySparkRuntimeError as e:
    print(f"❌ Error initializing Spark session: {e}")
    print("This often indicates an issue with the Java gateway.")
    print("Please try restarting the Colab runtime (Runtime -> Restart runtime) and re-running the cells.")
    # Set spark to None or handle the error appropriately for subsequent cells
    spark = None

❌ Error initializing Spark session: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.
This often indicates an issue with the Java gateway.
Please try restarting the Colab runtime (Runtime -> Restart runtime) and re-running the cells.


In [9]:
# CELL 3: Load Processed Data from Your Files
PROCESSED_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/processed'

# Find the most recent processed file
processed_files = glob.glob(f'{PROCESSED_DIR}/vendors_processed_*.csv')

if not processed_files:
    print("❌ No processed files found!")
    print("Please run Notebook 3 first to process the data.")
else:
    latest_file = __builtins__.max(processed_files, key=os.path.getctime)
    print(f"📁 Loading data from: {os.path.basename(latest_file)}\n")

    # Load CSV into Spark DataFrame with schema inference
    df_spark = spark.read.csv(
        latest_file,
        header=True,
        inferSchema=True,
        nullValue='',
        nanValue='NaN'
    )

    record_count = df_spark.count()
    print(f"✅ Loaded {record_count:,} vendors into Spark DataFrame\n")

    # Cache the DataFrame for better performance
    df_spark.cache()

    # Show schema
    print("="*100)
    print("DATA SCHEMA")
    print("="*100)
    df_spark.printSchema()

    # Show sample data
    print("\n" + "="*100)
    print("SAMPLE DATA (First 5 Records)")
    print("="*100)
    df_spark.select(
        'vendor_name', 'state', 'total_spend', 'vendor_tier', 'engagement_level'
    ).show(5, truncate=False)

📁 Loading data from: vendors_processed_20251105_064503.csv



AttributeError: 'NoneType' object has no attribute 'read'

In [7]:
# CELL 4: Data Quality & Completeness Analysis

# Ensure Spark session is active
spark = SparkSession.builder.getOrCreate()

print("\n" + "="*100)
print("DATA QUALITY ANALYSIS")
print("="*100 + "\n")

# 1. Null/Empty Value Analysis
print("1. Data Completeness by Column:")
print("-" * 100)

# Calculate total rows
total_rows = df_spark.count()

# Create a list of aggregation expressions for null/empty counts
agg_exprs = []
for col_name in df_spark.columns:
    if isinstance(df_spark.schema[col_name].dataType, StringType):
        # Count nulls and empty strings for string columns
        agg_exprs.append(
            count(when(col(col_name).isNull() | (trim(col(col_name)) == ''), col_name)).alias(col_name)
        )
    else:
        # Count only nulls for non-string columns
        agg_exprs.append(
            count(when(col(col_name).isNull(), col_name)).alias(col_name)
        )

# Perform aggregation to get all counts
null_counts_row = df_spark.agg(*agg_exprs).collect()[0]

null_analysis = []
for col_name in df_spark.columns:
    null_count = null_counts_row[col_name]
    filled_count = total_rows - null_count
    completeness = (filled_count / total_rows * 100) if total_rows > 0 else 0

    null_analysis.append({
        'column': col_name,
        'total': total_rows,
        'filled': filled_count,
        'empty': null_count,
        'completeness_%': __builtins__.round(completeness, 2) # Use built-in round
    })

null_df = pd.DataFrame(null_analysis).sort_values('completeness_%', ascending=False)
print(null_df.to_string(index=False))

# 2. Overall Completeness Score
total_cells = len(df_spark.columns) * total_rows
filled_cells = null_df['filled'].sum()
overall_completeness = (filled_cells / total_cells * 100) if total_cells > 0 else 0

print(f"\n📊 Overall Data Completeness Score: {overall_completeness:.2f}%")

# 3. Key Field Completeness
print("\n2. Critical Field Completeness:")
print("-" * 100)
critical_fields = ['vendor_name', 'bedrock_id', 'primary_email', 'phone',
                   'total_spend', 'state', 'vendor_type']
for field in critical_fields:
    if field in null_df['column'].values:
        completeness = null_df[null_df['column'] == field]['completeness_%'].values[0]
        status = "✅" if completeness >= 80 else "⚠️" if completeness >= 50 else "❌"
        print(f"  {status} {field:<30} {completeness:>6.2f}%")
    else:
        print(f"  ❓ {field:<30} N/A (Column not found in analysis)")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# CELL 5: Advanced Metrics Calculation
print("\n" + "="*100)
print("CALCULATING ADVANCED ANALYTICS METRICS")
print("="*100 + "\n")

# Ensure numeric columns are properly typed
df_spark = df_spark.withColumn('total_spend', col('total_spend').cast('double'))
df_spark = df_spark.withColumn('spend_2024', col('spend_2024').cast('double'))
df_spark = df_spark.withColumn('spend_2023', col('spend_2023').cast('double'))
df_spark = df_spark.withColumn('spend_2022', col('spend_2022').cast('double'))

# 1. Performance Score (0-100)
df_spark = df_spark.withColumn(
    'performance_score',
    (
        # Spend component (40 points)
        when(col('total_spend') >= 1000000, lit(40))
        .when(col('total_spend') >= 100000, lit(30))
        .when(col('total_spend') >= 10000, lit(20))
        .when(col('total_spend') > 0, lit(10))
        .otherwise(lit(0)) +

        # Growth component (30 points)
        when(col('spend_growth_2023_2024') >= 50, lit(30))
        .when(col('spend_growth_2023_2024') >= 20, lit(20))
        .when(col('spend_growth_2023_2024') >= 0, lit(10))
        .when(col('spend_growth_2023_2024') >= -20, lit(5))
        .otherwise(lit(0)) +

        # Activity component (30 points)
        when(col('activity_score') >= 75, lit(30))
        .when(col('activity_score') >= 50, lit(20))
        .when(col('activity_score') >= 25, lit(10))
        .otherwise(lit(0))
    )
)

print("✓ Performance scores calculated (0-100)")

# 2. Vendor Health Status
df_spark = df_spark.withColumn(
    'vendor_health_status',
    when((col('total_spend') > 0) & (col('spend_growth_2023_2024') >= 0) & (col('is_active') == 1), 'Healthy')
    .when((col('total_spend') > 0) & (col('spend_growth_2023_2024') < 0) & (col('spend_growth_2023_2024') >= -20), 'At Risk')
    .when((col('total_spend') > 0) & (col('spend_growth_2023_2024') < -20), 'Critical')
    .when(col('total_spend') == 0, 'Inactive')
    .otherwise('Unknown')
)

print("✓ Vendor health status calculated")

# 3. Spend per Quarter estimate
df_spark = df_spark.withColumn(
    'avg_quarterly_spend_2024',
    col('spend_2024') / 4
)

print("✓ Quarterly spend estimates calculated")

# 4. Contact quality score
df_spark = df_spark.withColumn(
    'contact_quality_score',
    (
        when(col('primary_email') != '', lit(50)).otherwise(lit(0)) +
        when(col('phone') != '', lit(50)).otherwise(lit(0))
    )
)

print("✓ Contact quality scores calculated")

print("\n✅ All advanced metrics calculated!\n")

In [ ]:
# CELL 6: Comprehensive Aggregations & Statistics
print("="*100)
print("AGGREGATIONS & BUSINESS INTELLIGENCE METRICS")
print("="*100 + "\n")

# 1. Top Level Summary
print("1. OVERALL VENDOR PORTFOLIO SUMMARY")
print("-" * 100)

summary_stats = df_spark.agg(
    count('*').alias('total_vendors'),
    countDistinct('state').alias('states_covered'),
    sum(when(col('total_spend') > 0, 1).otherwise(0)).alias('active_vendors'),
    sum('total_spend').alias('total_portfolio_spend'),
    avg('total_spend').alias('avg_spend_per_vendor'),
    max('total_spend').alias('max_vendor_spend'),
    min(when(col('total_spend') > 0, col('total_spend'))).alias('min_vendor_spend'),
    sum('spend_2024').alias('ytd_2024_spend'),
    sum('spend_2023').alias('ytd_2023_spend'),
    avg('performance_score').alias('avg_performance_score'),
    avg('activity_score').alias('avg_activity_score')
).collect()[0]

summary_df = pd.DataFrame([summary_stats.asDict()])
for key, value in summary_stats.asDict().items():
    if 'spend' in key.lower() or 'portfolio' in key.lower():
        print(f"  {key:<40} ${value:>20,.2f}")
    else:
        print(f"  {key:<40} {value:>20,.2f}" if isinstance(value, float) else f"  {key:<40} {value:>20}")

# 2. Vendor Tier Distribution
print("\n2. VENDOR TIER DISTRIBUTION")
print("-" * 100)

tier_dist = df_spark.groupBy('vendor_tier') \
    .agg(
        count('*').alias('vendor_count'),
        sum('total_spend').alias('tier_spend'),
        avg('total_spend').alias('avg_spend'),
        avg('performance_score').alias('avg_performance'),
        countDistinct('state').alias('states')
    ) \
    .orderBy(desc('tier_spend'))

tier_df = tier_dist.toPandas()
print(tier_df.to_string(index=False))

# 3. Geographic Analysis - Top States
print("\n3. GEOGRAPHIC ANALYSIS - TOP 15 STATES")
print("-" * 100)

state_analysis = df_spark.filter(col('state') != '') \
    .groupBy('state') \
    .agg(
        count('*').alias('vendor_count'),
        sum('total_spend').alias('total_spend'),
        avg('total_spend').alias('avg_spend'),
        sum('spend_2024').alias('spend_2024'),
        avg('performance_score').alias('avg_performance')
    ) \
    .orderBy(desc('total_spend')) \
    .limit(15)

state_df = state_analysis.toPandas()
print(state_df.to_string(index=False))

# 4. US Regional Analysis
print("\n4. REGIONAL ANALYSIS (US Regions)")
print("-" * 100)

if 'us_region' in df_spark.columns:
    region_analysis = df_spark.filter(col('us_region') != '') \
        .groupBy('us_region') \
        .agg(
            count('*').alias('vendor_count'),
            sum('total_spend').alias('total_spend'),
            avg('total_spend').alias('avg_spend'),
            sum('spend_2024').alias('spend_2024'),
            countDistinct('state').alias('states')
        ) \
        .orderBy(desc('total_spend'))

    region_df = region_analysis.toPandas()
    print(region_df.to_string(index=False))

# 5. Vendor Type Analysis
print("\n5. VENDOR TYPE ANALYSIS")
print("-" * 100)

type_analysis = df_spark.filter(col('vendor_type') != '') \
    .groupBy('vendor_type') \
    .agg(
        count('*').alias('vendor_count'),
        sum('total_spend').alias('total_spend'),
        avg('total_spend').alias('avg_spend'),
        avg('performance_score').alias('avg_performance')
    ) \
    .orderBy(desc('vendor_count')) \
    .limit(10)

type_df = type_analysis.toPandas()
print(type_df.to_string(index=False))

# 6. Payment Terms Analysis
print("\n6. PAYMENT TERMS ANALYSIS")
print("-" * 100)

payment_analysis = df_spark.filter(col('payment_terms') != '') \
    .groupBy('payment_terms') \
    .agg(
        count('*').alias('vendor_count'),
        sum('total_spend').alias('total_spend'),
        avg('total_spend').alias('avg_spend')
    ) \
    .orderBy(desc('vendor_count')) \
    .limit(10)

payment_df = payment_analysis.toPandas()
print(payment_df.to_string(index=False))

# 7. Vendor Health Status Distribution
print("\n7. VENDOR HEALTH STATUS DISTRIBUTION")
print("-" * 100)

health_analysis = df_spark.groupBy('vendor_health_status') \
    .agg(
        count('*').alias('vendor_count'),
        sum('total_spend').alias('total_spend'),
        avg('performance_score').alias('avg_performance')
    ) \
    .orderBy(desc('vendor_count'))

health_df = health_analysis.toPandas()
print(health_df.to_string(index=False))

# 8. Risk Category Analysis
print("\n8. RISK CATEGORY ANALYSIS")
print("-" * 100)

risk_analysis = df_spark.groupBy('risk_category') \
    .agg(
        count('*').alias('vendor_count'),
        sum('total_spend').alias('total_spend'),
        avg('spend_growth_2023_2024').alias('avg_growth_rate'),
        avg('performance_score').alias('avg_performance')
    ) \
    .orderBy(desc('vendor_count'))

risk_df = risk_analysis.toPandas()
print(risk_df.to_string(index=False))


In [ ]:
# CELL 7: Top and Bottom Performers
print("\n" + "="*100)
print("TOP & BOTTOM PERFORMERS ANALYSIS")
print("="*100 + "\n")

# Top 15 Vendors by Total Spend
print("1. TOP 15 VENDORS BY TOTAL SPEND")
print("-" * 100)

top_vendors = df_spark.orderBy(desc('total_spend')) \
    .select(
        'vendor_name', 'state', 'vendor_type', 'total_spend',
        'spend_2024', 'performance_score', 'vendor_health_status'
    ) \
    .limit(15)

top_df = top_vendors.toPandas()
print(top_df.to_string(index=False))

# Top 10 Growing Vendors
print("\n2. TOP 10 FASTEST GROWING VENDORS (2023-2024)")
print("-" * 100)

growing_vendors = df_spark.filter(col('spend_2023') > 0) \
    .orderBy(desc('spend_growth_2023_2024')) \
    .select(
        'vendor_name', 'state', 'spend_2023', 'spend_2024',
        'spend_growth_2023_2024', 'vendor_health_status'
    ) \
    .limit(10)

growing_df = growing_vendors.toPandas()
print(growing_df.to_string(index=False))

# Top 10 Declining Vendors
print("\n3. TOP 10 VENDORS AT RISK (Highest Decline)")
print("-" * 100)

declining_vendors = df_spark.filter(col('spend_2023') > 0) \
    .orderBy('spend_growth_2023_2024') \
    .select(
        'vendor_name', 'state', 'spend_2023', 'spend_2024',
        'spend_growth_2023_2024', 'vendor_health_status'
    ) \
    .limit(10)

declining_df = declining_vendors.toPandas()
print(declining_df.to_string(index=False))

# Highest Performance Score
print("\n4. TOP 10 HIGHEST PERFORMANCE SCORES")
print("-" * 100)

top_performance = df_spark.orderBy(desc('performance_score')) \
    .select(
        'vendor_name', 'state', 'total_spend', 'performance_score',
        'activity_score', 'engagement_level'
    ) \
    .limit(10)

perf_df = top_performance.toPandas()
print(perf_df.to_string(index=False))

In [ ]:
# CELL 8: Spend Trend Analysis
print("\n" + "="*100)
print("SPEND TREND ANALYSIS (2022-2024)")
print("="*100 + "\n")

# Year-over-year spend
yoy_stats = df_spark.agg(
    sum('spend_2022').alias('spend_2022'),
    sum('spend_2023').alias('spend_2023'),
    sum('spend_2024').alias('spend_2024')
).collect()[0]

spend_2022 = yoy_stats['spend_2022'] or 0
spend_2023 = yoy_stats['spend_2023'] or 0
spend_2024 = yoy_stats['spend_2024'] or 0

growth_2022_2023 = ((spend_2023 - spend_2022) / spend_2022 * 100) if spend_2022 > 0 else 0
growth_2023_2024 = ((spend_2024 - spend_2023) / spend_2023 * 100) if spend_2023 > 0 else 0

print(f"Year-over-Year Spend Analysis:")
print(f"  2022 Total Spend: ${spend_2022:>20,.2f}")
print(f"  2023 Total Spend: ${spend_2023:>20,.2f}  (Growth: {growth_2022_2023:>7.2f}%)")
print(f"  2024 Total Spend: ${spend_2024:>20,.2f}  (Growth: {growth_2023_2024:>7.2f}%)")


In [ ]:
# CELL 9: Export Enriched Data for Visualization
print("\n" + "="*100)
print("EXPORTING ENRICHED DATA FOR VISUALIZATION")
print("="*100 + "\n")

EXPORT_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Export main enriched dataset
enriched_file = f'{EXPORT_DIR}/vendors_enriched.csv'
df_spark.toPandas().to_csv(enriched_file, index=False)
print(f"✓ Exported enriched dataset: {os.path.basename(enriched_file)}")

# 2. Export tier distribution
tier_df.to_csv(f'{EXPORT_DIR}/tier_distribution.csv', index=False)
print(f"✓ Exported tier distribution")

# 3. Export state analysis
state_df.to_csv(f'{EXPORT_DIR}/state_analysis.csv', index=False)
print(f"✓ Exported state analysis")

# 4. Export top vendors
top_df.to_csv(f'{EXPORT_DIR}/top_vendors.csv', index=False)
print(f"✓ Exported top vendors")

# 5. Export health status
health_df.to_csv(f'{EXPORT_DIR}/vendor_health_status.csv', index=False)
print(f"✓ Exported vendor health status")

print(f"\n📁 All files exported to: {EXPORT_DIR}")

print("\n" + "="*100)
print("✅ PYSPARK PROCESSING COMPLETE!")
print("="*100)
print("\n📝 Next Steps:")
print("   1. ✅ Data scraped from website")
print("   2. ✅ Data loaded into SQLite database")
print("   3. ✅ Big data processing with PySpark")
print("   4. ➡️  Next: Open Notebook 5 for Interactive Visualizations & Dashboards")
print("="*100 + "\n")